# P2B API Testing

In [ ]:
import base64
import hashlib
import hmac
import json
import time
from typing import Dict, Any

import requests

## Configuration

In [ ]:
base_url = "https://api.p2pb2b.com"
market = "GGEZ1_USDT"

In [ ]:
# Accounts — uncomment the one you want to use

# Default account
api_key = "xx"
secret_key = "xx"

# MUTAZ account
# api_key = "xx"
# secret_key = "xx"

# MOHD account
# api_key = "xx"
# secret_key = "xx"

## Helper Functions

In [ ]:
def _generate_signature(payload: str) -> str:
    return hmac.new(secret_key.encode("utf8"), payload.encode("utf8"), hashlib.sha512).hexdigest()


def _build_auth(end_point: str, data: dict) -> tuple:
    """Inject request/nonce, return (json_data, headers)."""
    data["request"] = end_point
    data["nonce"] = int(time.time() * 1000)
    json_data = json.dumps(data, separators=(",", ":"))
    encrypted_data = base64.b64encode(json_data.encode("utf8")).decode("utf8")
    signature = _generate_signature(encrypted_data)
    headers = {
        "Content-Type": "application/json",
        "X-TXC-APIKEY": api_key,
        "X-TXC-PAYLOAD": encrypted_data,
        "X-TXC-SIGNATURE": signature,
    }
    return json_data, headers


def api_post(end_point: str, data: dict) -> dict:
    json_data, headers = _build_auth(end_point, data)
    response = requests.post(base_url + end_point, headers=headers, data=json_data)
    response.raise_for_status()
    return response.json()


def api_get(end_point: str, params: dict = None) -> dict:
    params = params or {}
    response = requests.get(base_url + end_point, params=params)
    response.raise_for_status()
    return response.json()


def pp(data):
    """Pretty print JSON."""
    print(json.dumps(data, indent=4))

## Account Balance

In [ ]:
balances = api_post("/api/v2/account/balances", {})["result"]
for asset_name in balances:
    if balances[asset_name]["available"] == "0" and balances[asset_name]["freeze"] == "0":
        continue
    print(f"Asset: {asset_name} , Available: {balances[asset_name]['available']}, Freeze: {balances[asset_name]['freeze']}")

## Active Orders

In [ ]:
active_orders = api_post("/api/v2/orders", {"market": market, "limit": 50})
pp(active_orders)

## Place Order

In [ ]:
order_data = {
    "market": market,
    "side": "buy",
    "amount": "80",
    "price": "0.088010",
}
result = api_post("/api/v2/order/new", order_data)
pp(result)

## Cancel Orders

### Cancel all active orders

In [ ]:
# Cancel all active orders for the market
orders_to_cancel = api_post("/api/v2/orders", {"market": market, "limit": 50})

for order in orders_to_cancel["result"]:
    result = api_post("/api/v2/order/cancel", {"market": market, "orderId": order["orderId"]})
    pp(result)
    time.sleep(1)

### Cancel 1 order

In [ ]:
order_id = 256198053319
result = api_post("/api/v2/order/cancel", {"market": market, "orderId": order_id})
pp(result)

## Order Info

In [ ]:
order_id = 256198053319
order_info = api_post("/api/v2/account/order", {"orderId": order_id})
pp(order_info)

## Trade History

In [ ]:
# Executed deals for a market (deprecated — use deal history below)
executed = api_post("/api/v2/account/executed_history", {"market": market})
pp(executed)

## Order History

In [ ]:
# Filled/partially filled orders — max 24h window, last 3 months
number_of_days = 1
time_now = int(time.time())
order_history = api_post("/api/v2/account/market_order_history", {
    "market": market,
    "startTime": time_now - (86400 * number_of_days),
    "endTime": time_now - (86400 * (number_of_days - 1)),
})
pp(order_history)

## Deal History

In [ ]:
# Deal history — max 24h window, last 3 months
number_of_days = 1
time_now = int(time.time())
deal_history = api_post("/api/v2/account/market_deal_history", {
    "market": market,
    "startTime": time_now - (86400 * number_of_days),
    "endTime": time_now - (86400 * (number_of_days - 1)),
})
pp(deal_history)

## Order Book

In [ ]:
depth = api_get("/api/v2/public/depth/result", {"market": market})
pp(depth)

## Market Data (Tickers)

In [ ]:
# Single ticker
ticker = api_get("/api/v2/public/ticker", {"market": market})
pp(ticker)

In [ ]:
# All tickers
all_tickers = api_get("/api/v2/public/tickers")
pp(all_tickers)

## Market Info

In [ ]:
markets = api_get("/api/v2/public/markets")
pp(markets)

## Klines

In [ ]:
klines = api_get("/api/v2/public/market/kline", {
    "market": market,
    "interval": "1m",
    "limit": 100,
})
pp(klines)

## Generate Order Book

In [ ]:
orders = [
    # Sell orders
    {"amount": "5000.00", "price": str(0.08775), "side": "sell"},
    {"amount": "5000.00", "price": str(0.0925), "side": "sell"},
    {"amount": "5000.00", "price": str(0.092), "side": "sell"},
    {"amount": "5000.00", "price": str(0.0915), "side": "sell"},
    {"amount": "5000.00", "price": str(0.091), "side": "sell"},
    {"amount": "10000.0", "price": str(0.0905), "side": "sell"},
    {"amount": "10000.0", "price": str(0.09), "side": "sell"},
    {"amount": "10000.0", "price": str(0.0895), "side": "sell"},
    {"amount": "12000.0", "price": str(0.089), "side": "sell"},
    {"amount": "14000.0", "price": str(0.0885), "side": "sell"},
    {"amount": "16000.0", "price": str(0.088), "side": "sell"},
    # Buy orders
    {"amount": "5000.00", "price": str(0.08725), "side": "buy"},
    {"amount": "5000.00", "price": str(0.08675), "side": "buy"},
    {"amount": "5000.00", "price": str(0.08625), "side": "buy"},
    {"amount": "5000.00", "price": str(0.08575), "side": "buy"},
    {"amount": "5000.00", "price": str(0.08525), "side": "buy"},
    {"amount": "10000.0", "price": str(0.08475), "side": "buy"},
    {"amount": "10000.0", "price": str(0.08425), "side": "buy"},
    {"amount": "10000.0", "price": str(0.08375), "side": "buy"},
    {"amount": "12000.0", "price": str(0.08325), "side": "buy"},
    {"amount": "14000.0", "price": str(0.08275), "side": "buy"},
    {"amount": "16000.0", "price": str(0.08225), "side": "buy"},
]


In [ ]:

for order in orders:
    result = api_post("/api/v2/order/new", {
        "market": market,
        "side": order["side"],
        "amount": order["amount"],
        "price": order["price"],
    })
    pp(result)
    time.sleep(1)